In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn import datasets, model_selection, metrics
import pandas as pd
import os


current_dir = os.getcwd()

if current_dir == '/mnt/datavisor':
    os.chdir("Elina_Proj/Sofi")
    print(f"Changed directory to: {os.getcwd()}")
elif os.path.basename(current_dir) == "Sofi":
    print("This notebook is in the correct directory")
else:
    print(f"Current working directory is: {current_dir}")

Changed directory to: /mnt/datavisor/Elina_Proj/Sofi


In [24]:
# Save 
!python3 /mnt/datavisor/pydatatools/archive_project.py --project_name=Elina_Proj

/srv/conda/envs/notebook/lib/python3.9/site-packages/google/api_core/_python_version_support.py:234: FutureWarning: You are using a non-supported Python version (3.9.13). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
INFO:__main__:compress success !
INFO:__main__:upload success!


In [9]:
# Define S3 and local file paths
s3_path = 's3://datavisor-trial-sofipoc-awsuswest2preprod/userstats_20250701_20250714_fixCD_36_v13/20250714/userstats-badtext-features-V2-groupId_userId_table.20250714_235959/part-00000/'
local_dir = 'Data/20250701_20250714_UML/features-V2-groupId_userId_table/'
local_file = os.path.join(local_dir, 'part-00000.csv')

# Ensure local directory exists
os.makedirs(local_dir, exist_ok=True)

# Read the S3 file as a tab-delimited file with no header, only the three columns specified.
try:
    df = pd.read_csv(
        s3_path,
        sep='\t',
        header=None,
        usecols=[0, 1, 2],
        names=["group_id", "user_id", "eleven"],
        on_bad_lines='skip',
        engine='python'
    )
except Exception as e:
    print(f"Error reading TSV from S3: {e}")
    df = None

if df is not None:
    df.to_csv(local_file, index=False, header=True)  # Write column headers
    print(f"Downloaded file from {s3_path} as CSV with headers and saved to {local_file}")
else:
    print("DataFrame not created due to errors in reading the file.")

Downloaded file from s3://datavisor-trial-sofipoc-awsuswest2preprod/userstats_20250701_20250714_fixCD_36_v13/20250714/userstats-badtext-features-V2-groupId_userId_table.20250714_235959/part-00000/ as CSV with headers and saved to Data/20250701_20250714_UML/features-V2-groupId_userId_table/part-00000.csv


In [25]:
import pandas as pd

# --- USER LEVEL METRICS ---

# All unique fraud user_ids (true positives)
fraud_label_file = 'Data/20250701_20250714_UML/fraud_label/sofi_uml_20250701-20250831_Redflages_V3_fraudLabel.csv'
try:
    df_fraud_label = pd.read_csv(fraud_label_file)
    # Strip leading/trailing spaces from 'user_id'
    if 'user_id' in df_fraud_label.columns:
        df_fraud_label['user_id'] = df_fraud_label['user_id'].astype(str).str.strip()
    print(f"Fraud label file loaded successfully with shape: {df_fraud_label.shape}")
except Exception as e:
    print(f"Error loading fraud label file: {e}")
    df_fraud_label = None

fraud_users = set(df_fraud_label['user_id'].unique()) if df_fraud_label is not None else set()

# Read model detected users from features table
detected_features_file = 'Data/20250701_20250714_UML/features-V2-groupId_userId_table/part-00000.csv'
try:
    df_features = pd.read_csv(detected_features_file)
    # Strip leading/trailing spaces from 'user_id'
    if 'user_id' in df_features.columns:
        df_features['user_id'] = df_features['user_id'].astype(str).str.strip()
    detected_users = set(df_features['user_id'].unique())
except Exception as e:
    print(f"Error loading detected features file: {e}")
    detected_users = set()

# True Positives (TP): Fraud user detected
tp_user = len(fraud_users & detected_users)
# False Positives (FP): Non-fraud user detected as fraud
fp_user = len(detected_users - fraud_users)
# False Negatives (FN): Fraud user not detected
fn_user = len(fraud_users - detected_users)
# True Negatives (TN): Non-fraud user not detected (optional)

total_users = len(fraud_users | detected_users)  # all users appearing in either list

# Detection Rate: % of total users flagged (for all users in universe)
detection_rate_user = len(detected_users) / total_users if total_users > 0 else float('nan')
# Precision: Of detected users, how many are truly fraud?
precision_user = tp_user / (tp_user + fp_user) if (tp_user + fp_user) > 0 else float('nan')
# Recall: Of all fraud users, how many were detected?
recall_user = tp_user / (tp_user + fn_user) if (tp_user + fn_user) > 0 else float('nan')
# Hurt Ratio: FP / (TP + FP), rate of non-frauds among detected
hurt_ratio_user = fp_user / (tp_user + fp_user) if (tp_user + fp_user) > 0 else float('nan')

print('User-level metrics:')
print(f'- Detection rate: {detection_rate_user * 100:.4f}%')
print(f'- Precision:      {precision_user * 100:.4f}%')
print(f'- Recall:         {recall_user * 100:.4f}%')
print(f'- Hurt ratio:     {hurt_ratio_user * 100:.4f}%')
print(f'- TP: {tp_user}, FP: {fp_user}, FN: {fn_user}, Total detected: {len(detected_users)}, Total fraud: {len(fraud_users)}')

Fraud label file loaded successfully with shape: (45235, 10)
User-level metrics:
- Detection rate: 0.0160
- Precision:      0.2005
- Recall:         0.0032
- Hurt ratio:     0.7995
- TP: 147, FP: 586, FN: 45088, Total detected: 733, Total fraud: 45235
